## How to fetch temp data

1. Uncomment the next two cells.
2. Run the 2nd cell with all variables on 1(Change the variable name too eg: df_b{1} <- Change inside the bracket), wait a couple minutes.
3. Run again but with the next number
4. Congrats you have all the temp data for all hexagons

In [1]:
import requests
import pandas as pd
import time
import math

# Configuration
MACRO_BATCH_SIZE = 500  # How many rows YOU want to process at once
MICRO_BATCH_SIZE = 100  # How many rows sent to API per call (API Limit)

def fetch_weather_manual_batch(df_all_hexes, batch_number):
    """
    Fetches weather for a specific slice of the dataframe.
    batch_number: 1-based index (1, 2, 3...)
    """
    # 1. Calculate the Slice Indices
    start_idx = (batch_number - 1) * MACRO_BATCH_SIZE
    end_idx = start_idx + MACRO_BATCH_SIZE
    
    # Get the subset of data for this manual run
    df_batch = df_all_hexes.iloc[start_idx : end_idx].copy()
    
    if df_batch.empty:
        print(f"⚠️ Batch {batch_number} is empty. (You might be done!)")
        return pd.DataFrame()

    print(f"⚡ Processing Batch {batch_number} (Rows {start_idx} to {min(end_idx, len(df_all_hexes))})...")

    # Prepare lists for API
    lats = df_batch['lat'].tolist()
    lons = df_batch['lon'].tolist()
    ids = df_batch['hex_id'].tolist()
    
    weather_data = []
    
    # 2. Internal Micro-Batching (Chunks of 100)
    for i in range(0, len(df_batch), MICRO_BATCH_SIZE):
        # Slicing the lists
        chunk_ids = ids[i : i + MICRO_BATCH_SIZE]
        chunk_lats = lats[i : i + MICRO_BATCH_SIZE]
        chunk_lons = lons[i : i + MICRO_BATCH_SIZE]
        
        params = {
            "latitude": ",".join(map(str, chunk_lats)),
            "longitude": ",".join(map(str, chunk_lons)),
            "daily": "temperature_2m_mean",
            "past_days": 30,
            "timezone": "auto"
        }
        
        try:
            r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10)
            r.raise_for_status()
            responses = r.json()
            
            if not isinstance(responses, list): responses = [responses]
            
            for hex_id, resp in zip(chunk_ids, responses):
                daily_temps = resp.get('daily', {}).get('temperature_2m_mean', [])
                if daily_temps:
                    valid = [t for t in daily_temps if t is not None]
                    avg_temp = sum(valid) / len(valid) if valid else float('nan')
                else:
                    avg_temp = float('nan')
                
                weather_data.append({'hex_id': hex_id, 'local_temp_c': avg_temp})
            
            print(f"   ... Sub-batch {i//MICRO_BATCH_SIZE + 1} complete.")
            time.sleep(0.5) # Short pause between API calls
            
        except Exception as e:
            print(f"   ❌ Error on sub-batch: {e}")

    print(f"✅ Batch {batch_number} Finished.")
    return pd.DataFrame(weather_data)

In [2]:
# # Check how many batches you need
# total_hexes = len(hex_df)
# total_batches = math.ceil(total_hexes / MACRO_BATCH_SIZE)
# print(f"You have {total_hexes} hexagons.")
# print(f"You need to run Batches 1 through {total_batches}.")

# # Container to store results
# if 'all_weather_data' not in locals():
#     all_weather_data = []

In [3]:
# # --- BATCH ---
# df_b4 = fetch_weather_manual_batch(hex_df, batch_number=4)
# all_weather_data.append(df_b4)

In [4]:
#all_weather_data_df = pd.concat(all_weather_data, ignore_index=True)

In [5]:
#all_weather_data_df.to_csv('hex_weather_data_all.csv', index=False)